<h2 align="center">Car Damage Detection Project: Hyperparameter Tunning</h2>

In [1]:
import os
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import time
import torchvision.models as models
from matplotlib import pyplot as plt
import optuna

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

### Load Data

In [3]:
image_transforms = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [4]:
dataset_path = "./dataset"

dataset = datasets.ImageFolder(root=dataset_path, transform=image_transforms)
len(dataset)

2300

In [5]:
2300*0.75

1725.0

In [6]:
class_names = dataset.classes
class_names 

['F_Breakage', 'F_Crushed', 'F_Normal', 'R_Breakage', 'R_Crushed', 'R_Normal']

In [7]:
num_classes = len(dataset.classes)
num_classes

6

In [8]:
train_size = int(0.75*len(dataset))
val_size = len(dataset) - train_size

train_size, val_size

(1725, 575)

In [9]:
from torch.utils.data import random_split

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

In [10]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=True)

### Model Training & Hyperparameter Tunning

In [11]:
# Load the pre-trained ResNet model
class CarClassifierResNet(nn.Module):
    def __init__(self, num_classes, dropout_rate=0.5):
        super().__init__()
        self.model = models.resnet50(weights='DEFAULT')
        # Freeze all layers except the final fully connected layer
        for param in self.model.parameters():
            param.requires_grad = False
            
        # Unfreeze layer4 and fc layers
        for param in self.model.layer4.parameters():
            param.requires_grad = True            
            
        # Replace the final fully connected layer
        self.model.fc = nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(self.model.fc.in_features, num_classes)
        )

    def forward(self, x):
        x = self.model(x)
        return x

In [12]:
# Define the objective function for Optuna
def objective(trial):
    # Suggest values for the hyperparameters
    lr = trial.suggest_float('lr', 1e-5, 1e-2, log=True)
    dropout_rate = trial.suggest_float('dropout_rate', 0.2, 0.7)
    
    # Load the model
    model = CarClassifierResNet(num_classes=num_classes, dropout_rate=dropout_rate).to(device)
    
    # Define the loss function and optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    
    # Training loop (using fewer epochs for faster hyperparameter tuning)
    epochs = 3
    start = time.time()
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for batch_num, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * images.size(0)

        epoch_loss = running_loss / len(train_loader.dataset)
        
        # Validation loop
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        accuracy = 100 * correct / total
        
        # Report intermediate result to Optuna
        trial.report(accuracy, epoch)
        
        # Handle pruning (if applicable)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    end = time.time()
    print(f"Execution time: {end - start} seconds")
    
    return accuracy

In [13]:
# Create the study and optimize
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20)

[I 2026-11-17 16:40:40,023] A new study created in memory with name: no-name-2d19d85e-c216-4782-a16a-5dd183d75853
[I 2026-11-17 16:47:09,656] Trial 0 finished with value: 77.3913043478261 and parameters: {'lr': 0.0002895478966496597, 'dropout_rate': 0.3407801820641687}. Best is trial 0 with value: 77.3913043478261.


Execution time: 387.61087703704834 seconds


[I 2026-11-17 16:53:14,107] Trial 1 finished with value: 79.47826086956522 and parameters: {'lr': 0.0007895089797271251, 'dropout_rate': 0.41328583818409803}. Best is trial 1 with value: 79.47826086956522.


Execution time: 363.3656897544861 seconds


[I 2026-11-17 16:59:20,964] Trial 2 finished with value: 78.78260869565217 and parameters: {'lr': 0.00089301900412623, 'dropout_rate': 0.4705257564485301}. Best is trial 1 with value: 79.47826086956522.


Execution time: 365.8654305934906 seconds


... (5 outputs left out)

Execution time: 249.57809925079346 seconds


[I 2026-11-17 17:19:31,714] Trial 10 pruned. 
[I 2026-11-17 17:22:05,351] Trial 11 pruned. 
[I 2026-11-17 17:25:55,426] Trial 12 finished with value: 74.95652173913044 and parameters: {'lr': 0.0016964204751624516, 'dropout_rate': 0.2596367842438206}. Best is trial 9 with value: 81.04347826086956.


Execution time: 229.7146439552307 seconds


[I 2026-11-17 17:27:12,024] Trial 13 pruned. 
[I 2026-11-17 17:28:28,563] Trial 14 pruned. 
[I 2026-11-17 17:29:45,976] Trial 15 pruned. 
[I 2026-11-17 17:32:19,750] Trial 16 pruned. 
[I 2026-11-17 17:34:53,062] Trial 17 pruned. 
[I 2026-11-17 17:36:09,384] Trial 18 pruned. 
[I 2026-11-17 17:37:25,851] Trial 19 pruned. 


In [14]:
study.best_params

{'lr': 0.0021067579520305762, 'dropout_rate': 0.44509963349664716}

I ran it earlier and I got the best accuracy at lr = 0.005 and dropout_rate = 0.2, so I will be using that